# Exploration du corpus français FQuADRetrieval

Ce notebook vérifie le corpus, génère les embeddings et affiche les documents les plus proches d'une requête française. Les données ont été téléchargées depuis Hugging Face dans `data/raw/fquad_retrieval`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
DATASET_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fquad_retrieval'
assert DATASET_DIR.exists(), f'Corpus introuvable : {DATASET_DIR}'
print(f'Projet : {PROJECT_ROOT}')
print(f'Corpus : {DATASET_DIR}')

Projet : /Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel
Corpus : /Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/data/raw/fquad_retrieval


In [2]:
corpus = pd.concat([
    pd.read_parquet(DATASET_DIR / 'corpus' / 'validation-00000-of-00001.parquet'),
    pd.read_parquet(DATASET_DIR / 'corpus' / 'test-00000-of-00001.parquet'),
], ignore_index=True)
queries = pd.concat([
    pd.read_parquet(DATASET_DIR / 'queries' / 'validation-00000-of-00001.parquet'),
    pd.read_parquet(DATASET_DIR / 'queries' / 'test-00000-of-00001.parquet'),
], ignore_index=True)
print(f'{len(corpus)} documents ; {len(queries)} requêtes')
corpus.head(3)

366 documents ; 500 requêtes


,_id,text,title
0,pégase_23_55,Le patient que Wilhelm Stekel évoque dans son ...,pégase_23_55
1,tension-transitoire-de-rétablissement_17_8,L'évolution temporelle de la tension sur la bo...,tension-transitoire-de-rétablissement_17_8
2,sadi-carnot-(physicien)_12_19,Nicolas Léonard Sadi Carnot est né à Paris au ...,sadi-carnot-(physicien)_12_19


In [3]:
from src.embeddings import EmbeddingEncoder

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encoder = EmbeddingEncoder(MODEL_NAME)
documents = (corpus['title'].fillna('') + '. ' + corpus['text']).tolist()
corpus_embeddings = encoder.encode(documents, batch_size=32)
print('Dimension des vecteurs :', encoder.dimension)
print('Matrice des embeddings :', corpus_embeddings.shape)

/Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 199/19


Dimension des vecteurs : 384
Matrice des embeddings : (366, 384)


/Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/src/embeddings/encoder.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return int(self.model.get_sentence_embedding_dimension())


In [4]:
def rechercher(query: str, limite: int = 5) -> pd.DataFrame:
    query_embedding = encoder.encode(query)
    scores = corpus_embeddings @ query_embedding
    resultats = corpus.iloc[scores.argsort()[::-1][:limite]].copy()
    resultats.insert(0, 'score', scores[scores.argsort()[::-1][:limite]])
    return resultats[['score', '_id', 'title', 'text']]

rechercher('Quelle est la capitale de la France ?', limite=5)

,score,_id,title,text
23,0.587340,histoire-de-la-bretagne_18_22,histoire-de-la-bretagne_18_22,"Politiquement, la région est à contre-courant ..."
231,0.587340,histoire-de-la-bretagne_18_22,histoire-de-la-bretagne_18_22,"Politiquement, la région est à contre-courant ..."
197,0.501966,pierre-lambert-de-la-motte_2_65,pierre-lambert-de-la-motte_2_65,L'année 1648 est marquée par de graves trouble...
57,0.498813,pierre-lambert-de-la-motte_2_30,pierre-lambert-de-la-motte_2_30,La ville de Caen est l'une des plus religieuse...
174,0.473843,histoire-de-la-bretagne_18_106,histoire-de-la-bretagne_18_106,La Troisième République a des difficultés à s'...


## Suite

Modifiez la requête précédente pour contrôler qualitativement la pertinence. Utilisez ensuite `evaluation.ipynb` pour mesurer automatiquement Recall@k et MRR.